<h1 style=\"text-align: center; font-size: 50px;\"> Register Model </h1>

# Notebook Overview

- Start Execution
- Install and Import Libraries
- Configure Settings
- Register the Model Log Results to MLFlow

# Start Execution

In [1]:
import logging
import time

# Configure logger
logger: logging.Logger = logging.getLogger("register_model_logger")
logger.setLevel(logging.INFO)
logger.propagate = False  # Prevent duplicate logs from parent loggers

# Set formatter
formatter: logging.Formatter = logging.Formatter(
    fmt="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

# Configure and attach stream handler
stream_handler: logging.StreamHandler = logging.StreamHandler()
stream_handler.setFormatter(formatter)
logger.addHandler(stream_handler)

In [2]:
start_time = time.time()  
logger.info("Notebook execution started.")

2025-08-21 14:51:28 - INFO - Notebook execution started.


# Install and Import Libraries

In [3]:
%%time

%pip install -r ../requirements.txt --quiet

Note: you may need to restart the kernel to use updated packages.
CPU times: user 29.4 ms, sys: 1.4 ms, total: 30.8 ms
Wall time: 2.39 s


In [ ]:
import nemo                             # NVIDIA NeMo core package
import nemo.collections.asr as nemo_asr # Speech Recognition (ASR) collection
import nemo.collections.tts as nemo_tts # Text-to-Speech (TTS) collection

# ------------------------- Transformers -------------------------
from transformers import MarianMTModel, MarianTokenizer

# ------------------------- Audio Processing Utilities -------------------------
import IPython                          # For playing audio inside Jupyter Notebooks
import soundfile                        # For reading and writing audio files
from pathlib import Path                # Filesystem path management

# ------------------------- System Utilities -------------------------

import os                               # Operating system interfaces
import shutil                           # High-level file operations
import uuid                             # Unique ID generation
import io                               # Input/Output core tools
import base64                           # Encoding and decoding base64 strings
import json                             # JSON serialization and deserialization
import warnings                         # Suppressing and managing warnings
import numpy as np                      # Numerical array operations
np.float_ = np.float64
import torch

# ------------------------- MLflow Integration -------------------------

import mlflow                           # MLflow experiment tracking and model management
from mlflow.types.schema import Schema, ColSpec
from mlflow.types import ParamSchema, ParamSpec
from mlflow.models import ModelSignature

# ------------------------ Utils Import ------------------------
import sys
sys.path.append("../src")
from onnx_utils import ModelExportConfig
from utils import load_config
from mlflow import Logger

[NeMo W 2025-08-21 14:51:35 nemo_logging:349] /opt/conda/envs/aistudio/lib/python3.10/site-packages/_distutils_hack/__init__.py:53: UserWarning: Reliance on distutils from stdlib is deprecated. Users must rely on setuptools to provide the distutils module. Avoid importing distutils or import setuptools first, and avoid setting SETUPTOOLS_USE_DISTUTILS=stdlib. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
      warnings.warn(
    


# Configure Settings

In [5]:
# ------------------------ Suppress Verbose Logs ------------------------
warnings.filterwarnings("ignore")

# Suppress NeMo internal logging
logging.getLogger('nemo_logger').setLevel(logging.ERROR)

In [6]:
# ------------------------- Model File Paths -------------------------
MT_MODEL = "Helsinki-NLP/opus-mt-en-es"
ASR_MODEL_PATH = "/home/jovyan/datafabric/STT_En_Citrinet_1024_Gamma_0.25/stt_en_citrinet_1024_gamma_0_25.nemo"                  # Speech-to-Text (ASR) model
SPECTROGRAM_GENERATOR_PATH = "/home/jovyan/datafabric/TTS_Es_Multispeaker_FastPitch_HiFiGAN/tts_es_fastpitch_multispeaker.nemo"  # Spectrogram generator model (FastPitch)
VOCODER_PATH = "/home/jovyan/datafabric/TTS_Es_Multispeaker_FastPitch_HiFiGAN/tts_es_hifigan_ft_fastpitch_multispeaker.nemo"     # Vocoder model (HiFiGAN)

AUDIO_SAMPLE_PATH = "../data/ForrestGump.mp3"      # Path to the input English audio sample

# ------------------------- MLflow Experiment Configuration -------------------------

EXPERIMENT_NAME = "NeMo_Translation_Experiment"    # MLflow experiment name
RUN_NAME = "NeMo_en_es_Translation_Run"            # Specific run name inside the experiment
MODEL_NAME = "nemo_en_es"                          # Registered model name in MLflow
DEMO_PATH = "../demo"                              # Path to save demo outputs

# Register the Model and Log Results to MLFlow

In [ ]:
# ------------------------- Model Signature Definition -------------------------

# Define input/output schema for MLflow model signature
input_schema = Schema([
    ColSpec("string", "source_text"),
    ColSpec("string", "source_serialized_audio"),
])

output_schema = Schema([
    ColSpec("string", "original_text"),
    ColSpec("string", "translated_text"),
    ColSpec("string", "translated_serialized_audio"),
])

params_schema = ParamSchema([
    ParamSpec("use_audio", "boolean", False)
])

# Create model signature
model_signature = ModelSignature(
    inputs=input_schema,
    outputs=output_schema,
    params=params_schema
)

In [ ]:
# ------------------------- MLflow Model Logging and Registration -------------------------

mlflow.set_tracking_uri('/phoenix/mlflow')
# Set the MLflow experiment
mlflow.set_experiment(experiment_name=EXPERIMENT_NAME)

# Start a new MLflow run
with mlflow.start_run(run_name=RUN_NAME) as run:
    # Define the set of NeMo model components to be logged
    nemo_model_artifacts = {
        "enc_dec_CTC": ASR_MODEL_PATH,
        "fast_pitch": SPECTROGRAM_GENERATOR_PATH,
        "hifi_gan": VOCODER_PATH,
    }

    # Loading Models in memory to convert to onnx
    mt_model = MarianMTModel.from_pretrained(MT_MODEL)
    asr_model = nemo_asr.models.EncDecCTCModel.restore_from(nemo_model_artifacts["enc_dec_CTC"])
    fast_pitch_model = nemo_tts.models.FastPitchModel.restore_from(nemo_model_artifacts["fast_pitch"])
    hifi_gan_model = nemo_tts.models.HifiGanModel.restore_from(nemo_model_artifacts["hifi_gan"])

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 🎯 Create ModelExportConfig objects with loaded models (manual configuration)
    model_configs = [ 
        ModelExportConfig(
            model=mt_model,                         # 🚀 Pre-loaded Transformers model!
            model_name="Helsinki-NLP",              # ONNX file naming
            task="translation",                     # Model task
        ),
        # NeMo ASR model
        ModelExportConfig(
            model=asr_model.to(device),                        # 🚀 Pre-loaded NeMo ASR model!
            model_name="enc_dec_CTC",               # ONNX file naming
        ),
        # NeMo FastPitch model
        ModelExportConfig(
            model=fast_pitch_model.to(device),                 # 🚀 Pre-loaded NeMo TTS model!
            model_name="fast_pitch",                # ONNX file naming
        ),
        # NeMo HifiGAN model
        ModelExportConfig(
            model=hifi_gan_model.to(device),                   # 🚀 Pre-loaded NeMo Vocoder model!
            model_name="hifi_gan",                  # ONNX file naming
        ),
    ]

    # Log the custom translation model using new Logger
    Logger.log_model(
        model_name=MODEL_NAME,
        nemo_models=nemo_model_artifacts,
        demo_folder="../demo",
        config_path="../configs/config.yaml",
        pip_requirements="../requirements.txt",
        signature=model_signature,
        models_to_convert_onnx=model_configs
    )

    # Register the logged model in MLflow Model Registry
    mlflow.register_model(
        model_uri=f"runs:/{run.info.run_id}/{MODEL_NAME}",
        name=MODEL_NAME
    )

W0821 14:51:39.830279 139865642697152 file_store.py:333] Malformed experiment 'tmp'. Detailed error Yaml file '/phoenix/mlflow/tmp/meta.yaml' does not exist.
Traceback (most recent call last):
  File "/opt/conda/envs/aistudio/lib/python3.10/site-packages/mlflow/store/tracking/file_store.py", line 329, in search_experiments
    exp = self._get_experiment(exp_id, view_type)
  File "/opt/conda/envs/aistudio/lib/python3.10/site-packages/mlflow/store/tracking/file_store.py", line 427, in _get_experiment
    meta = FileStore._read_yaml(experiment_dir, FileStore.META_DATA_FILE_NAME)
  File "/opt/conda/envs/aistudio/lib/python3.10/site-packages/mlflow/store/tracking/file_store.py", line 1373, in _read_yaml
    return _read_helper(root, file_name, attempts_remaining=retries)
  File "/opt/conda/envs/aistudio/lib/python3.10/site-packages/mlflow/store/tracking/file_store.py", line 1366, in _read_helper
    result = read_yaml(root, file_name)
  File "/opt/conda/envs/aistudio/lib/python3.10/site-pac

Removing weight norm...
Removing weight norm...


2025-08-21 14:52:26 - INFO - Model saved to ONNX: hifi_gan.onnx
2025-08-21 14:52:26 - INFO - ✅ Converted hifi_gan: hifi_gan.onnx
2025-08-21 14:52:26 - INFO - 📦 Added ONNX artifact: onnx_Helsinki-NLP -> Helsinki-NLP.onnx
2025-08-21 14:52:26 - INFO - 📦 Added ONNX artifact: onnx_enc_dec_CTC -> enc_dec_CTC.onnx
2025-08-21 14:52:26 - INFO - 📦 Added ONNX artifact: onnx_fast_pitch -> fast_pitch.onnx
2025-08-21 14:52:26 - INFO - 📦 Added ONNX artifact: onnx_hifi_gan -> hifi_gan.onnx
2025-08-21 14:52:26 - INFO -   Creating individual model directories for artifacts...
2025-08-21 14:52:26 - INFO -   Copying ONNX models for 4 models: ['Helsinki-NLP', 'enc_dec_CTC', 'fast_pitch', 'hifi_gan']
2025-08-21 14:52:26 - INFO - 📄 Model file already has correct name: Helsinki-NLP.onnx
2025-08-21 14:52:26 - INFO - 📄 Model file already has correct name: enc_dec_CTC.onnx
2025-08-21 14:52:26 - INFO - 📄 Model file already has correct name: fast_pitch.onnx
2025-08-21 14:52:26 - INFO - 📄 Model file already has cor

2025-08-21 14:52:47 - INFO - Model logged with artifacts: ['model', 'demo', 'config', 'onnx_Helsinki-NLP', 'onnx_enc_dec_CTC', 'onnx_fast_pitch', 'onnx_hifi_gan', 'model_Helsinki-NLP', 'model_enc_dec_CTC', 'model_fast_pitch', 'model_hifi_gan']
2025-08-21 14:52:47 - INFO - ✅ Model logged with 4 model directories created!
Registered model 'nemo_en_es' already exists. Creating a new version of this model...
Created version '3' of model 'nemo_en_es'.


In [9]:
# ------------------------- Success Confirmation -------------------------

print(f"✅ Model '{MODEL_NAME}' successfully logged and registered under experiment '{EXPERIMENT_NAME}'.")

✅ Model 'nemo_en_es' successfully logged and registered under experiment 'NeMo_Translation_Experiment'.


In [10]:
end_time: float = time.time()
elapsed_time: float = end_time - start_time
elapsed_minutes: int = int(elapsed_time // 60)
elapsed_seconds: float = elapsed_time % 60

logger.info(f"⏱️ Total execution time: {elapsed_minutes}m {elapsed_seconds:.2f}s")
logger.info("✅ Notebook execution completed successfully.")

2025-08-21 14:52:47 - INFO - ⏱️ Total execution time: 1m 19.08s
2025-08-21 14:52:47 - INFO - ✅ Notebook execution completed successfully.


Built with ❤️ using [**HP AI Studio**](https://hp.com/ai-studio).